In [2]:
import pandas as pd
import json
import os
import re

# =============================================================================
# 1. 파일 경로 설정 (사용자 요청 반영)
# =============================================================================
# 이전 단계에서 생성한 '키워드 없고 초록만 있는' 파일
target_csv_path = 'keyword_missing_abstract_being_papers.csv'
# 룰맵 파일
rule_map_path = 'keyword_rule_map.json'
# 결과 저장 파일
output_csv_path = 'extract_keywords_from_abstract.csv'

# =============================================================================
# 2. 룰맵 로딩 및 정규화
# =============================================================================
print("1. 키워드 룰맵 로드 및 정규화(공백제거) 중...")

normalized_rule_map = {} 

with open(rule_map_path, 'r', encoding='utf-8') as f:
    rule_data = json.load(f)

# 리스트나 딕셔너리 구조에 맞춰 유연하게 로드
items = []
if isinstance(rule_data, dict):
    if "exact_map" in rule_data:
        for v, c in rule_data["exact_map"].items():
            norm_key = v.replace(" ", "").lower()
            normalized_rule_map[norm_key] = c
    # 추가 리스트가 있다면
    if "variants_list" in rule_data:
        items = rule_data["variants_list"]
elif isinstance(rule_data, list):
    items = rule_data

for item in items:
    if "variant_norm" in item and "chosen" in item:
        norm_key = str(item["variant_norm"]).replace(" ", "").lower()
        normalized_rule_map[norm_key] = item["chosen"]

print(f"   -> 룰맵 준비 완료: {len(normalized_rule_map)}개 항목")

# =============================================================================
# 3. 조사 처리 로직이 추가된 추출 함수
# =============================================================================
# 흔히 붙는 조사 목록 (끝 글자 기준)
COMMON_JOSA = ['은', '는', '이', '가', '을', '를', '의', '에', '와', '과', '로', '으로', '도', '만', '서']

def extract_keywords_with_josa_handling(text, rule_map, max_ngram=5):
    if not isinstance(text, str) or not text:
        return ""

    # 특수문자 제거 (한글, 영문, 숫자 외 공백 처리)
    clean_text = re.sub(r'[^\w\s가-힣]', ' ', text)
    words = clean_text.split()
    
    found_keywords = set()
    n = len(words)
    i = 0
    
    while i < n:
        match_found = False
        
        # Longest Match (긴 단어 조합부터 검사)
        for length in range(min(max_ngram, n - i), 0, -1):
            chunk_words = words[i : i+length]
            base_chunk_str = "".join(chunk_words).lower() # 공백 제거+소문자
            
            # 1. 정확히 매칭되는지 확인 (예: "deeplearning", "인공지능")
            if base_chunk_str in rule_map:
                found_keywords.add(rule_map[base_chunk_str])
                i += length
                match_found = True
                break
            
            # 2. [추가된 로직] 한글이라면 조사를 떼고 확인 (예: "인공지능을" -> "인공지능")
            # 마지막 글자가 조사인지 확인하고 제거 시도
            if len(base_chunk_str) > 1: # 최소 2글자 이상일 때만
                last_char = base_chunk_str[-1]
                if last_char in COMMON_JOSA:
                    stripped_str = base_chunk_str[:-1] # 조사 제거
                    if stripped_str in rule_map:
                        found_keywords.add(rule_map[stripped_str])
                        i += length
                        match_found = True
                        break
                
                # '으로' 같은 2글자 조사 처리
                if len(base_chunk_str) > 2 and base_chunk_str.endswith('으로'):
                     stripped_str = base_chunk_str[:-2]
                     if stripped_str in rule_map:
                        found_keywords.add(rule_map[stripped_str])
                        i += length
                        match_found = True
                        break

        if not match_found:
            i += 1
            
    return ",".join(list(found_keywords))

# =============================================================================
# 4. 실행 및 저장
# =============================================================================
print("2. 논문 초록에서 키워드 추출 (조사 처리 적용)...")

if os.path.exists(target_csv_path):
    df = pd.read_csv(target_csv_path)
    
    df['ABST_KR'] = df['ABST_KR'].fillna("")
    df['ABST_EN'] = df['ABST_EN'].fillna("")
    
    extracted_results = []
    
    for idx, row in df.iterrows():
        # 국문과 영문을 모두 사용
        full_text = str(row['ABST_KR']) + " " + str(row['ABST_EN'])
        
        # 개선된 함수 사용
        kwd = extract_keywords_with_josa_handling(full_text, normalized_rule_map)
        extracted_results.append(kwd)
        
    df['EXTRACTED_KEYWORDS'] = extracted_results
    
    # 추출된 내용이 있는 경우만 필터링해서 볼까요? (선택사항)
    # 여기서는 원본 유지 (추출 안 된 것도 포함해서 저장)
    
    df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    
    # 결과 통계
    non_empty = df[df['EXTRACTED_KEYWORDS'] != ""]
    print("-" * 50)
    print(f"✅ 추출 완료!")
    print(f"   - 전체 대상: {len(df)}건")
    print(f"   - 키워드 발견 성공: {len(non_empty)}건")
    print(f"   - 저장 파일: {output_csv_path}")
    print("-" * 50)
    print("미리보기 (상위 3건):")
    print(df[['NODE_ID', 'EXTRACTED_KEYWORDS']].head(3))
else:
    print(f"❌ 오류: 입력 파일({target_csv_path})이 없습니다.")

1. 키워드 룰맵 로드 및 정규화(공백제거) 중...
   -> 룰맵 준비 완료: 1931개 항목
2. 논문 초록에서 키워드 추출 (조사 처리 적용)...
--------------------------------------------------
✅ 추출 완료!
   - 전체 대상: 1322건
   - 키워드 발견 성공: 1111건
   - 저장 파일: extract_keywords_from_abstract.csv
--------------------------------------------------
미리보기 (상위 3건):
        NODE_ID      EXTRACTED_KEYWORDS
0  NODE10565031  performance,simulation
1  NODE10561439                        
2  NODE10561440      autonomous driving


In [3]:
# 추출된 한/영 키워드 확인

import pandas as pd
from collections import Counter

# 1. 파일 경로 설정
input_csv_path = 'extract_keywords_from_abstract.csv'

# 2. 데이터 로드 및 키워드 집계
print("키워드 추출 결과를 분석합니다...")

if pd.io.common.file_exists(input_csv_path): # 파일 존재 여부 확인
    df = pd.read_csv(input_csv_path)
    
    # NaN 값을 빈 문자열로 처리
    df['EXTRACTED_KEYWORDS'] = df['EXTRACTED_KEYWORDS'].fillna("")
    
    # 모든 키워드를 하나의 리스트로 모으기
    all_keywords = []
    
    for kwd_str in df['EXTRACTED_KEYWORDS']:
        if kwd_str.strip(): # 빈 칸이 아닌 경우만
            # 쉼표(,)로 구분된 키워드들을 분리해서 리스트에 추가
            keywords = [k.strip() for k in kwd_str.split(",")]
            all_keywords.extend(keywords)
            
    # 개수 세기 (Counter 활용)
    keyword_counts = Counter(all_keywords)
    
    # 3. 결과 출력
    print(f"\n✅ 총 추출된 키워드 수 (중복 포함): {len(all_keywords)}개")
    print(f"✅ 발견된 고유 키워드 종류: {len(keyword_counts)}개")
    print("-" * 40)
    print(f"{'순위':<5} {'키워드':<30} {'개수':<5}")
    print("-" * 40)
    
    # 상위 20개 출력
    for rank, (keyword, count) in enumerate(keyword_counts.most_common(20), 1):
        print(f"{rank:<5} {keyword:<30} {count:<5}")
        
    print("-" * 40)
    
    # (선택 사항) 전체 통계를 파일로 저장하고 싶다면 아래 주석 해제
    # result_df = pd.DataFrame(keyword_counts.most_common(), columns=['Keyword', 'Count'])
    # result_df.to_csv('keyword_statistics.csv', index=False, encoding='utf-8-sig')
    # print("전체 키워드 통계가 'keyword_statistics.csv'로 저장되었습니다.")

else:
    print(f"❌ 오류: '{input_csv_path}' 파일이 없습니다. 이전 단계를 먼저 실행해주세요.")

키워드 추출 결과를 분석합니다...

✅ 총 추출된 키워드 수 (중복 포함): 3293개
✅ 발견된 고유 키워드 종류: 423개
----------------------------------------
순위    키워드                            개수   
----------------------------------------
1     performance                    175  
2     machine learning               164  
3     artificial intelligence        132  
4     research trends                117  
5     efficiency                     101  
6     internet of things             73   
7     metaverse                      68   
8     deep learning                  59   
9     ict                            58   
10    blockchain                     56   
11    5g                             56   
12    autonomous driving             51   
13    content                        48   
14    optimization                   47   
15    6g                             46   
16    carbon neutrality              46   
17    generation                     42   
18    covid 19                       39   
19    network                

In [4]:
# 추출된 한글 키워드 확인

import pandas as pd
import json
import os
import re

# =============================================================================
# 1. 파일 경로 설정
# =============================================================================
result_csv_path = 'extract_keywords_from_abstract.csv' # 결과 파일
rule_map_path = 'keyword_rule_map.json' # 룰맵 파일

# =============================================================================
# 2. 데이터 로드 및 룰맵 준비
# =============================================================================
print("1. 데이터 및 룰맵 로드 중...")

# [룰맵 로드]
normalized_korean_rules = {} # 한글 규칙만 저장 { '인공지능': 'machine learning' }

with open(rule_map_path, 'r', encoding='utf-8') as f:
    rule_data = json.load(f)

# 룰맵 파싱 (이전과 동일 로직)
items = []
if isinstance(rule_data, dict):
    if "exact_map" in rule_data:
        for v, c in rule_data["exact_map"].items():
            # 한글이 포함된 키만 필터링
            if re.search('[가-힣]', v):
                norm_key = v.replace(" ", "").lower()
                normalized_korean_rules[norm_key] = c
    if "variants_list" in rule_data:
        items = rule_data["variants_list"]
elif isinstance(rule_data, list):
    items = rule_data

for item in items:
    if "variant_norm" in item and "chosen" in item:
        v = str(item["variant_norm"])
        if re.search('[가-힣]', v):
            norm_key = v.replace(" ", "").lower()
            normalized_korean_rules[norm_key] = item["chosen"]

print(f"   -> 검증할 한글 규칙 수: {len(normalized_korean_rules)}개")

# [결과 파일 로드]
if os.path.exists(result_csv_path):
    df = pd.read_csv(result_csv_path)
    df['ABST_KR'] = df['ABST_KR'].fillna("")
    df['EXTRACTED_KEYWORDS'] = df['EXTRACTED_KEYWORDS'].fillna("")
    
    # =============================================================================
    # 3. 검증 로직 수행
    # =============================================================================
    print("2. 한글 키워드 변환 여부 검증 시작...")
    
    match_count = 0
    sample_logs = []
    
    # 흔한 조사 제거용 정규식 (간단 버전)
    josa_pattern = re.compile(r'(은|는|이|가|을|를|의|에|로|으로|와|과|도|만|서)$')

    for idx, row in df.iterrows():
        abst_kr = str(row['ABST_KR'])
        extracted = str(row['EXTRACTED_KEYWORDS'])
        
        if not abst_kr.strip(): continue # 국문 초록 없으면 패스
        
        # 정규화된 초록 (공백 제거) - 검색 편의상
        norm_abst = re.sub(r'\s+', '', abst_kr)
        
        # 한글 룰 하나씩 대조 (속도 위해 일부만 샘플링하거나 전체 루프)
        # 여기서는 정확성을 위해 루프를 돌지만, 데이터가 많으면 오래 걸릴 수 있음
        
        row_hit = False
        for kr_key, en_val in normalized_korean_rules.items():
            # 초록에 한글 키워드가 포함되어 있는지 확인 (단순 포함 관계)
            # 주의: "인공지능"이 "인공지능을" 안에 포함되므로 in 연산자로 확인 가능
            if kr_key in norm_abst:
                # 결과에 영문 변환값이 있는지 확인
                if en_val in extracted:
                    match_count += 1
                    row_hit = True
                    
                    # 로그용 샘플 저장 (최대 10개)
                    if len(sample_logs) < 10:
                        sample_logs.append({
                            'NODE_ID': row['NODE_ID'],
                            'Found_Korean': kr_key,
                            'Converted_To': en_val,
                            'In_Abstract': (abst_kr[:30] + "..."),
                            'Extracted_Result': extracted
                        })
                    break # 한 행에서 하나라도 찾으면 카운트하고 다음 행으로 (중복 카운트 방지)
    
    # =============================================================================
    # 4. 결과 출력
    # =============================================================================
    print("-" * 60)
    print(f"✅ 검증 완료!")
    print(f"   - 한글 단어가 포함된 논문 중 변환 성공한 케이스: {match_count}건")
    print("-" * 60)
    print("[변환 성공 샘플 미리보기]")
    print(f"{'Korean(In Abstract)':<20} | {'Converted(English)':<30} | {'Extracted Result'}")
    print("-" * 60)
    
    for log in sample_logs:
        print(f"{log['Found_Korean']:<20} -> {log['Converted_To']:<30} | {log['Extracted_Result'][:30]}...")
        
else:
    print(f"❌ 오류: '{result_csv_path}' 파일이 없습니다.")

1. 데이터 및 룰맵 로드 중...
   -> 검증할 한글 규칙 수: 378개
2. 한글 키워드 변환 여부 검증 시작...
------------------------------------------------------------
✅ 검증 완료!
   - 한글 단어가 포함된 논문 중 변환 성공한 케이스: 861건
------------------------------------------------------------
[변환 성공 샘플 미리보기]
Korean(In Abstract)  | Converted(English)             | Extracted Result
------------------------------------------------------------
시뮬레이션                -> simulation                     | performance,simulation...
자율주행                 -> autonomous driving             | autonomous driving...
신호시스템                -> signal system                  | autonomous driving,research tr...
자율주행                 -> autonomous driving             | autonomous driving...
사물인터넷                -> internet of things             | internet of things...
교통사고                 -> traffic safety                 | traffic safety,ict...
드론                   -> drone                          | internet of things,drone,netwo...
블록체인                 -> blockch

In [5]:
# 추출되지 않은 논문 5개 확인

import pandas as pd
import json
import os
import re

# =============================================================================
# 1. 파일 경로 설정 (사용자 요청 반영)
# =============================================================================
# 이전 단계에서 생성한 '키워드 없고 초록만 있는' 파일
target_csv_path = 'keyword_missing_abstract_being_papers.csv'
# 룰맵 파일
rule_map_path = 'keyword_rule_map.json'
# 결과 저장 파일
output_csv_path = 'extract_keywords_from_abstract.csv'

# =============================================================================
# 2. 룰맵 로딩 및 정규화
# =============================================================================
print("1. 키워드 룰맵 로드 및 정규화(공백제거) 중...")

normalized_rule_map = {} 

with open(rule_map_path, 'r', encoding='utf-8') as f:
    rule_data = json.load(f)

# 리스트나 딕셔너리 구조에 맞춰 유연하게 로드
items = []
if isinstance(rule_data, dict):
    if "exact_map" in rule_data:
        for v, c in rule_data["exact_map"].items():
            norm_key = v.replace(" ", "").lower()
            normalized_rule_map[norm_key] = c
    # 추가 리스트가 있다면
    if "variants_list" in rule_data:
        items = rule_data["variants_list"]
elif isinstance(rule_data, list):
    items = rule_data

for item in items:
    if "variant_norm" in item and "chosen" in item:
        norm_key = str(item["variant_norm"]).replace(" ", "").lower()
        normalized_rule_map[norm_key] = item["chosen"]

print(f"   -> 룰맵 준비 완료: {len(normalized_rule_map)}개 항목")

# =============================================================================
# 3. 조사 처리 로직이 추가된 추출 함수
# =============================================================================
COMMON_JOSA = ['은', '는', '이', '가', '을', '를', '의', '에', '와', '과', '로', '으로', '도', '만', '서']

def extract_keywords_with_josa_handling(text, rule_map, max_ngram=5):
    if not isinstance(text, str) or not text:
        return ""

    # 특수문자 제거 (한글, 영문, 숫자 외 공백 처리)
    clean_text = re.sub(r'[^\w\s가-힣]', ' ', text)
    words = clean_text.split()
    
    found_keywords = set()
    n = len(words)
    i = 0
    
    while i < n:
        match_found = False
        
        # Longest Match (긴 단어 조합부터 검사)
        for length in range(min(max_ngram, n - i), 0, -1):
            chunk_words = words[i : i+length]
            base_chunk_str = "".join(chunk_words).lower() # 공백 제거+소문자
            
            # 1. 정확히 매칭되는지 확인
            if base_chunk_str in rule_map:
                found_keywords.add(rule_map[base_chunk_str])
                i += length
                match_found = True
                break
            
            # 2. 조사 처리 로직
            if len(base_chunk_str) > 1: 
                last_char = base_chunk_str[-1]
                if last_char in COMMON_JOSA:
                    stripped_str = base_chunk_str[:-1] 
                    if stripped_str in rule_map:
                        found_keywords.add(rule_map[stripped_str])
                        i += length
                        match_found = True
                        break
                
                if len(base_chunk_str) > 2 and base_chunk_str.endswith('으로'):
                     stripped_str = base_chunk_str[:-2]
                     if stripped_str in rule_map:
                        found_keywords.add(rule_map[stripped_str])
                        i += length
                        match_found = True
                        break

        if not match_found:
            i += 1
            
    return ",".join(list(found_keywords))

# =============================================================================
# 4. 실행 및 저장, 그리고 실패 샘플 확인
# =============================================================================
print("2. 논문 초록에서 키워드 추출 (조사 처리 적용)...")

if os.path.exists(target_csv_path):
    df = pd.read_csv(target_csv_path)
    
    df['ABST_KR'] = df['ABST_KR'].fillna("")
    df['ABST_EN'] = df['ABST_EN'].fillna("")
    
    extracted_results = []
    
    for idx, row in df.iterrows():
        full_text = str(row['ABST_KR']) + " " + str(row['ABST_EN'])
        kwd = extract_keywords_with_josa_handling(full_text, normalized_rule_map)
        extracted_results.append(kwd)
        
    df['EXTRACTED_KEYWORDS'] = extracted_results
    
    # 저장
    df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    
    # 통계 계산
    non_empty = df[df['EXTRACTED_KEYWORDS'] != ""]
    empty_df = df[df['EXTRACTED_KEYWORDS'] == ""] # 추출되지 않은 데이터만 필터링
    
    print("-" * 50)
    print(f"✅ 추출 완료!")
    print(f"   - 전체 대상: {len(df)}건")
    print(f"   - 키워드 발견 성공: {len(non_empty)}건")
    print(f"   - 키워드 미발견: {len(empty_df)}건")
    print(f"   - 저장 파일: {output_csv_path}")
    print("-" * 50)
    
    # 요청하신 실패 샘플 출력 부분
    if len(empty_df) > 0:
        print("\n" + "="*60)
        print(f"❌ [추출 실패 샘플 (상위 5건)]")
        print("   -> 룰맵에 있는 단어가 초록에 없거나 매칭되지 않은 경우입니다.")
        print("="*60)
        
        for i, row in empty_df.head(5).iterrows():
            node_id = row['NODE_ID']
            k_abst = str(row['ABST_KR'])
            e_abst = str(row['ABST_EN'])
            
            # 초록이 너무 길면 잘라서 보여줌
            k_preview = k_abst[:60] + "..." if len(k_abst) > 60 else k_abst
            e_preview = e_abst[:60] + "..." if len(e_abst) > 60 else e_abst
            
            print(f"🔹 NODE_ID: {node_id}")
            print(f"   - 국문: {k_preview}")
            print(f"   - 영문: {e_preview}")
            print("-" * 40)
    else:
        print("🎉 모든 논문에서 키워드가 성공적으로 추출되었습니다!")

else:
    print(f"❌ 오류: 입력 파일({target_csv_path})이 없습니다.")

1. 키워드 룰맵 로드 및 정규화(공백제거) 중...
   -> 룰맵 준비 완료: 1931개 항목
2. 논문 초록에서 키워드 추출 (조사 처리 적용)...
--------------------------------------------------
✅ 추출 완료!
   - 전체 대상: 1322건
   - 키워드 발견 성공: 1111건
   - 키워드 미발견: 211건
   - 저장 파일: extract_keywords_from_abstract.csv
--------------------------------------------------

❌ [추출 실패 샘플 (상위 5건)]
   -> 룰맵에 있는 단어가 초록에 없거나 매칭되지 않은 경우입니다.
🔹 NODE_ID: NODE10561439
   - 국문: 자율주행차의 등장으로 자동차의 안전 기준 개선 필요성이 증가하고 있다. 현재의 자동차 안전기준은 운전자가 차...
   - 영문: 
----------------------------------------
🔹 NODE_ID: NODE10561442
   - 국문: 국민 생활과 국가 경제에 밀접한 도로 교통 인프라는 도로 안전성 및 편의성 등 모빌리티에 필수적인 서비스로를...
   - 영문: 
----------------------------------------
🔹 NODE_ID: NODE10561446
   - 국문: 재난 및 안전과 관련된 사건은 우리 생활과 밀접한 관계가 있어서 많은 사람들이 공감하고 관심을 갖고 있지만 ...
   - 영문: 
----------------------------------------
🔹 NODE_ID: NODE10561449
   - 국문: 본고에서는 재난재해 발생 시 현장 대응의 미숙함으로 많은 인명 피해가 발생하는 등 인구 과밀화와 시설 집중이...
   - 영문: 
----------------------------------------
🔹 NODE_ID: NODE11179615
   - 국문: 갈수록 지능적이고